In [11]:
!pip install contractions

In [21]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 8.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tsfresh 0.21.0 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
thinc 8.3.6 requ

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("crowdflower/twitter-airline-sentiment")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/twitter-airline-sentiment


In [2]:
import pandas as pd
import nltk
import numpy as np

In [3]:
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_ru is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_r

True

In [22]:
df = pd.read_csv('/kaggle/input/twitter-airline-sentiment/Tweets.csv')

In [23]:
df.head()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [24]:
df.groupby('airline')['airline_sentiment'].value_counts()


airline         airline_sentiment
American        negative             1960
                neutral               463
                positive              336
Delta           negative              955
                neutral               723
                positive              544
Southwest       negative             1186
                neutral               664
                positive              570
US Airways      negative             2263
                neutral               381
                positive              269
United          negative             2633
                neutral               697
                positive              492
Virgin America  negative              181
                neutral               171
                positive              152
Name: count, dtype: int64

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14640 entries, 0 to 14639
Data columns (total 15 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   tweet_id                      14640 non-null  int64  
 1   airline_sentiment             14640 non-null  object 
 2   airline_sentiment_confidence  14640 non-null  float64
 3   negativereason                9178 non-null   object 
 4   negativereason_confidence     10522 non-null  float64
 5   airline                       14640 non-null  object 
 6   airline_sentiment_gold        40 non-null     object 
 7   name                          14640 non-null  object 
 8   negativereason_gold           32 non-null     object 
 9   retweet_count                 14640 non-null  int64  
 10  text                          14640 non-null  object 
 11  tweet_coord                   1019 non-null   object 
 12  tweet_created                 14640 non-null  object 
 13  t

In [26]:
null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df)) * 100

# Display columns with their null percentages
display(pd.DataFrame({'Null Count': null_counts, 'Null Percentage (%)': null_percentages}).sort_values(by='Null Percentage (%)', ascending=False))

,Null Count,Null Percentage (%)
negativereason_gold,14608,99.781421
airline_sentiment_gold,14600,99.726776
tweet_coord,13621,93.039617
negativereason,5462,37.308743
user_timezone,4820,32.923497
tweet_location,4733,32.329235
negativereason_confidence,4118,28.128415
tweet_id,0,0.000000
airline_sentiment,0,0.000000
airline_sentiment_confidence,0,0.000000


In [27]:
df = df[['text', 'airline_sentiment']]
df

,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative
...,...,...
14635,@AmericanAir thank you we got on a different f...,positive
14636,@AmericanAir leaving over 20 minutes Late Flig...,negative
14637,@AmericanAir Please bring American Airlines to...,neutral
14638,"@AmericanAir you have my money, you change my ...",negative


In [12]:
from nltk import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk import sent_tokenize
import contractions
import re
import string

In [15]:
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = contractions.fix(text)
    text = re.sub(r"http\S+|www.\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"#\w+", "", text)
    text = text.translate(str.maketrans('', '', string.punctuation))

    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'[^\x00-\x7F]+', '', text)

    tokens = nltk.word_tokenize(text.lower())
    lemmatized = [lemmatizer.lemmatize(token) for token in tokens]

    return lemmatized

In [28]:
df['text'] = df['text'].apply(lambda x: clean_text(x))
df

,text,airline_sentiment
0,"[what, said]",neutral
1,"[plus, you, have, added, commercial, to, the, ...",positive
2,"[i, did, not, today, must, mean, i, need, to, ...",neutral
3,"[it, is, really, aggressive, to, blast, obnoxi...",negative
4,"[and, it, is, a, really, big, bad, thing, abou...",negative
...,...,...
14635,"[thank, you, we, got, on, a, different, flight...",positive
14636,"[leaving, over, 20, minute, late, flight, no, ...",negative
14637,"[please, bring, american, airline, to]",neutral
14638,"[you, have, my, money, you, change, my, flight...",negative


In [17]:
import gensim.downloader as api
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [18]:
model = api.load('word2vec-google-news-300')

[==================================================] 100.0% 1662.8/1662.8MB downloaded


In [19]:
def avg_word2vec(arr, model):
    valid_vectors = [model[word] for word in arr if word in model]

    if not valid_vectors:
        return np.zeros(model.vector_size)

    return np.mean(valid_vectors, axis=0)

In [ ]:
df['text'] = df['text'].apply(lambda x: avg_word2vec(x, model))


In [30]:
df.loc[14,'text']

array([-9.27734375e-02,  8.10546875e-02, -1.00097656e-01,  2.07031250e-01,
        1.72119141e-02, -1.02050781e-01,  1.18164062e-01, -1.92382812e-01,
       -2.53906250e-02,  1.17187500e-01, -2.42919922e-02, -4.73022461e-03,
       -1.48437500e-01,  2.28271484e-02,  1.08398438e-01,  4.45556641e-03,
        4.57031250e-01,  1.66015625e-01,  1.37695312e-01,  1.33789062e-01,
        1.38671875e-01,  1.82617188e-01,  2.45117188e-01, -1.77734375e-01,
        1.90429688e-01, -1.50390625e-01, -6.10351562e-03,  6.83593750e-02,
       -5.09643555e-03,  4.78515625e-02,  4.73632812e-02, -1.62109375e-01,
       -1.33789062e-01,  2.63671875e-01,  1.17187500e-01,  1.90734863e-03,
       -1.90429688e-02, -8.30078125e-02,  1.15234375e-01,  2.07031250e-01,
        2.46093750e-01,  2.91748047e-02, -8.48388672e-03,  2.45117188e-01,
        3.35937500e-01, -1.73828125e-01, -7.47070312e-02, -2.79541016e-02,
        1.78710938e-01, -4.39453125e-02,  6.46972656e-03, -1.60156250e-01,
        1.68945312e-01,  

In [32]:
le = LabelEncoder()
df['airline_sentiment'] = le.fit_transform(df['airline_sentiment'])
df
# positive = 2, neutral = 1, negative = 0

,text,airline_sentiment
0,"[0.0652771, -0.025177002, 0.15722656, -0.00170...",1
1,"[0.025238037, -0.0074768066, -0.0210495, 0.082...",2
2,"[-0.01674028, 0.032792524, 0.059326172, 0.0763...",1
3,"[0.018322945, 0.063874125, 0.012535095, 0.0904...",0
4,"[0.102321625, 0.011981487, 0.059326172, 0.0765...",0
...,...,...
14635,"[-0.020568848, 0.014526367, -0.010360718, 0.17...",2
14636,"[0.009950183, 0.025126139, 0.02629162, 0.04133...",0
14637,"[-0.016540527, -0.0063476562, 0.08666992, 0.18...",1
14638,"[0.016659824, -0.010292747, 0.046623923, 0.160...",0


In [33]:
x = df['text']
y = df['airline_sentiment']

In [34]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [35]:
x_train.shape

(11712,)

In [36]:
log = LogisticRegression(verbose = 3)
log.fit(x_train.tolist(), y_train)

[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    2.3s finished


LogisticRegression(verbose=3)

In [37]:
y_pred = log.predict(x_test.tolist())
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.787568306010929


In [38]:
def predict_message_class(model, w2vmodel, message):
    message = clean_text(message)
    message_vector = avg_word2vec(message, w2vmodel)
    prediction = model.predict([message_vector])
    return prediction[0]

In [41]:
text1 = "Had a smooth flight with JetBlue. Loved the snacks and friendly staff!"
predict_message_class(log,model,text1)

2

In [48]:
text2 = "Checking in for my flight with United now. Boarding starts in an hour."
predict_message_class(log,model,text2)

0

In [45]:
text3 = "Terrible experience with Spirit Airlines — lost my luggage and rude staff."
predict_message_class(log,model,text3)

0